# Advanced Examples and Workflows

This notebook demonstrates complex queries, batch operations, error handling, and complete workflows combining employees and learning materials.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from teamup.database import get_session
from teamup.repository import (
    EmployeeRepository,
    LearningMaterialTypeRepository,
    LearningMaterialRepository
)
from faker import Faker
import json

fake = Faker()
print("✓ Imports ready")

## Example 1: Complete Onboarding Workflow

Create a new employee and assign them recommended learning materials.

In [ ]:
with get_session() as session:
    emp_repo = EmployeeRepository(session)
    mat_repo = LearningMaterialRepository(session)
    type_repo = LearningMaterialTypeRepository(session)
    
    # 1. Create new employee
    new_employee = emp_repo.create(
        fullname="Sarah Martinez",
        job_title="Junior Developer",
        age=24
    )
    print(f"1. Created employee: {new_employee}")
    
    # 2. Get or create "Book" type
    book_type = type_repo.get_by_name("Book")
    if not book_type:
        book_type = type_repo.create(name="Book")
    print(f"2. Using material type: {book_type}")
    
    # 3. Create onboarding materials
    materials = [
        {
            "name": "Company Handbook 2024",
            "description": "Essential company policies and procedures",
            "notes": json.dumps({"mandatory": True, "completion_days": 7})
        },
        {
            "name": "Git and GitHub Basics",
            "description": "Version control fundamentals",
            "notes": json.dumps({"difficulty": "beginner", "estimated_hours": 8})
        },
        {
            "name": "Python Best Practices",
            "description": "Clean code and testing in Python",
            "notes": json.dumps({"difficulty": "intermediate", "estimated_hours": 12})
        }
    ]
    
    print("\n3. Created learning materials:")
    for mat_data in materials:
        material = mat_repo.create(
            name=mat_data["name"],
            description=mat_data["description"],
            type_id=book_type.id,
            notes=mat_data["notes"]
        )
        print(f"   - {material.name}")
    
    print(f"\n✓ Onboarding complete for {new_employee.fullname}")

## Example 2: Career Progression Tracking

Track an employee's career growth with updated materials.

In [ ]:
with get_session() as session:
    emp_repo = EmployeeRepository(session)
    mat_repo = LearningMaterialRepository(session)
    
    # Find Sarah
    sarah = emp_repo.find_by_name("Sarah Martinez")
    if sarah:
        sarah = sarah[0]
        print(f"Current status: {sarah}")
        
        # After 6 months: promotion
        updated_sarah = emp_repo.update(
            sarah.id,
            job_title="Mid-level Developer"
        )
        print(f"\nAfter 6 months: {updated_sarah}")
        
        # Find advanced learning materials
        advanced_materials = mat_repo.find_by_name("Advanced")
        print(f"\nAdvanced materials available: {len(advanced_materials)}")
        for mat in advanced_materials[:3]:
            print(f"  - {mat.name}")

## Example 3: Batch Operations with Faker

Create multiple realistic test records efficiently.

In [ ]:
# Generate 10 employees with realistic data
with get_session() as session:
    emp_repo = EmployeeRepository(session)
    
    print("Creating 10 employees with Faker:")
    job_titles = [
        "Developer", "Designer", "Product Manager", 
        "Data Analyst", "QA Engineer"
    ]
    
    created_employees = []
    for i in range(10):
        emp = emp_repo.create(
            fullname=fake.name()[:50],
            job_title=fake.random_element(job_titles),
            age=fake.random_int(min=22, max=55)
        )
        created_employees.append(emp)
        print(f"  {i+1}. {emp.fullname} - {emp.job_title}")
    
    print(f"\n✓ Created {len(created_employees)} employees")

In [ ]:
# Generate 20 learning materials
with get_session() as session:
    mat_repo = LearningMaterialRepository(session)
    type_repo = LearningMaterialTypeRepository(session)
    
    # Get available types
    types = type_repo.get_all()
    if not types:
        print("No material types found. Creating defaults...")
        types = [
            type_repo.create(name="Book"),
            type_repo.create(name="Course"),
            type_repo.create(name="Video")
        ]
    
    print(f"\nCreating 20 learning materials using {len(types)} types:")
    
    topics = [
        "Python", "JavaScript", "DevOps", "Cloud Computing",
        "Machine Learning", "Database Design", "Security",
        "Agile Methods", "Leadership", "Communication"
    ]
    
    for i in range(20):
        topic = fake.random_element(topics)
        mat_type = fake.random_element(types)
        
        material = mat_repo.create(
            name=f"{topic} {mat_type.name}",
            description=fake.text(max_nb_chars=200),
            type_id=mat_type.id,
            price=fake.random_int(min=0, max=10000),
            notes=json.dumps({
                "difficulty": fake.random_element(["beginner", "intermediate", "advanced"]),
                "rating": round(fake.random.uniform(3.5, 5.0), 1)
            })
        )
        
        if (i + 1) % 5 == 0:
            print(f"  Created {i+1} materials...")
    
    print(f"\n✓ Total materials in database: {mat_repo.count()}")

## Example 4: Complex Search and Analytics

Combine multiple queries to generate insights.

In [ ]:
with get_session() as session:
    emp_repo = EmployeeRepository(session)
    mat_repo = LearningMaterialRepository(session)
    type_repo = LearningMaterialTypeRepository(session)
    
    # Employee statistics
    print("=== EMPLOYEE STATISTICS ===")
    total_employees = emp_repo.count()
    print(f"Total employees: {total_employees}")
    
    developers = emp_repo.find_by_job_title("Developer")
    print(f"Developers: {len(developers)}")
    
    young_employees = emp_repo.find_by_age_range(min_age=20, max_age=30)
    print(f"Employees aged 20-30: {len(young_employees)}")
    
    # Material statistics
    print("\n=== LEARNING MATERIAL STATISTICS ===")
    total_materials = mat_repo.count()
    print(f"Total materials: {total_materials}")
    
    # Group by type
    types = type_repo.get_all()
    print("\nMaterials by type:")
    for mat_type in types:
        materials_of_type = mat_repo.find_by_type(mat_type.id)
        print(f"  {mat_type.name}: {len(materials_of_type)}")
    
    # Price analysis
    free_materials = mat_repo.find_by_price_range(min_price=0, max_price=0)
    affordable = mat_repo.find_by_price_range(min_price=1, max_price=2500)
    expensive = mat_repo.find_by_price_range(min_price=5000, max_price=999999)
    
    print("\nPrice distribution:")
    print(f"  Free: {len(free_materials)}")
    print(f"  Affordable ($0.01-$25): {len(affordable)}")
    print(f"  Premium ($50+): {len(expensive)}")

## Example 5: Error Handling and Validation

Demonstrate proper error handling patterns.

In [ ]:
# Handling nonexistent records
with get_session() as session:
    emp_repo = EmployeeRepository(session)
    
    # Try to get nonexistent employee
    result = emp_repo.get_by_id(999999)
    if result is None:
        print("✓ Correctly returned None for nonexistent employee")
    
    # Try to update nonexistent employee
    updated = emp_repo.update(999999, fullname="Does Not Exist")
    if updated is None:
        print("✓ Correctly returned None when updating nonexistent employee")
    
    # Try to delete nonexistent employee
    deleted = emp_repo.delete(999999)
    if not deleted:
        print("✓ Correctly returned False when deleting nonexistent employee")

In [ ]:
# Transaction rollback demonstration
print("Testing transaction rollback:")

with get_session() as session:
    emp_repo = EmployeeRepository(session)
    initial_count = emp_repo.count()
    print(f"Initial employee count: {initial_count}")

try:
    with get_session() as session:
        emp_repo = EmployeeRepository(session)
        
        # Create employee
        emp = emp_repo.create(fullname="Temporary Employee", age=25)
        print(f"Created employee: {emp.fullname}")
        
        # Simulate an error
        raise ValueError("Simulated error - this will rollback the transaction")
        
except ValueError as e:
    print(f"Caught error: {e}")

# Verify rollback
with get_session() as session:
    emp_repo = EmployeeRepository(session)
    final_count = emp_repo.count()
    print(f"Final employee count: {final_count}")
    
    if final_count == initial_count:
        print("✓ Transaction correctly rolled back")
    else:
        print("✗ Transaction was not rolled back (unexpected)")

## Example 6: Working with Complex JSON

Store and retrieve nested JSON data in the notes field.

In [ ]:
with get_session() as session:
    mat_repo = LearningMaterialRepository(session)
    type_repo = LearningMaterialTypeRepository(session)
    
    # Get or create a type
    course_type = type_repo.get_by_name("Course")
    if not course_type:
        course_type = type_repo.create(name="Course")
    
    # Create material with complex nested JSON
    complex_notes = {
        "metadata": {
            "author": "Dr. Jane Smith",
            "published": "2024-01-15",
            "last_updated": "2024-03-10"
        },
        "content": {
            "modules": [
                {"id": 1, "title": "Introduction", "duration_hours": 2},
                {"id": 2, "title": "Advanced Topics", "duration_hours": 8},
                {"id": 3, "title": "Projects", "duration_hours": 10}
            ],
            "total_hours": 20
        },
        "requirements": {
            "prerequisites": ["Basic Python", "Git basics"],
            "software": ["Python 3.12+", "VS Code"],
            "hardware": "8GB RAM minimum"
        },
        "statistics": {
            "enrollments": 1543,
            "completion_rate": 0.87,
            "average_rating": 4.6
        }
    }
    
    material = mat_repo.create(
        name="Advanced Python Development",
        description="Comprehensive course on advanced Python topics",
        type_id=course_type.id,
        price=9900,  # $99.00
        notes=json.dumps(complex_notes, indent=2)
    )
    
    print(f"Created: {material.name}")
    print("\nStored JSON structure:")
    print(json.dumps(json.loads(material.notes), indent=2))

In [ ]:
# Query and parse the JSON
with get_session() as session:
    mat_repo = LearningMaterialRepository(session)
    
    materials = mat_repo.find_by_name("Advanced Python Development")
    if materials:
        material = materials[0]
        notes = json.loads(material.notes)
        
        print(f"Course: {material.name}")
        print(f"Author: {notes['metadata']['author']}")
        print(f"Total duration: {notes['content']['total_hours']} hours")
        print(f"Completion rate: {notes['statistics']['completion_rate'] * 100:.0f}%")
        print(f"\nModules:")
        for module in notes['content']['modules']:
            print(f"  {module['id']}. {module['title']} ({module['duration_hours']}h)")

## Example 7: Data Export

Export data for reporting or analysis.

In [ ]:
import csv
from io import StringIO

with get_session() as session:
    emp_repo = EmployeeRepository(session)
    
    # Get all employees
    employees = emp_repo.get_all()[:10]  # First 10
    
    # Create CSV
    output = StringIO()
    writer = csv.writer(output)
    
    # Write header
    writer.writerow(['ID', 'Full Name', 'Job Title', 'Age'])
    
    # Write data
    for emp in employees:
        writer.writerow([
            emp.id,
            emp.fullname,
            emp.job_title or 'N/A',
            emp.age or 'N/A'
        ])
    
    # Display CSV
    print("Employee Export (CSV):")
    print(output.getvalue())

## Summary

This notebook demonstrated:

1. **Complete workflows** - Onboarding and career progression
2. **Batch operations** - Efficient creation of multiple records
3. **Analytics** - Complex queries and statistics
4. **Error handling** - Proper validation and rollback
5. **JSON operations** - Complex nested data structures
6. **Data export** - CSV generation for reporting

## Best Practices

- Always use `with get_session()` for automatic cleanup
- Check for `None` when retrieving records
- Use transaction rollback for error recovery
- Store JSON as strings with `json.dumps()`
- Parse JSON with `json.loads()` when reading
- Respect field length constraints (50 for fullname, 30 for job_title, etc.)
- Use empty arrays `[]` for tag_ids and wiki_note_ids to avoid trigger validation